영상 하나에서 약 1초마다 프레임 추출하기  v  
프레임 15장을 한 화면에 표시하기         v  
YOLO로 각 프레임의 차량 종류·개수·박스 좌표 추출하기    v  
영상 이름 / 프레임 번호 / 시간 / 차량 수 형태의 표 만들기      v  
익숙해지면 연속 프레임에 객체 추적을 적용해 같은 차량의 이동 경로 확인하기  v

In [ ]:
from pathlib import Path

videos = sorted(
    p for p in Path("/kaggle/input/datasets/snmahsa/driving-test/driving").rglob("*")
    # rglot : 폴더 안의 모든 하위 폴더 포함 파일/폴더를 하나씩 반환 
    if p.is_file() and p.suffix.lower() == '.mp4'
)

print(f"MP4 데이터 개수 : {len(videos)}")

In [ ]:
# 영상 하나에서 약 1초마다 프레임 추출하기 
    # 영상의 FPS를 확인, 
    # 그만큼의 프레임마다 한 장씩 추출 
import cv2
import matplotlib.pyplot as plt

video_path = videos[0]
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("영상을 열 수 없습니다.")

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    cap.release()
    raise ValueError("FPS 를 확인할 수 없습니다")

frames = []
times = []

frame_idx = 0
next_second = 0

try:
    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_idx >= round(next_second * fps):
            frames.append(frame)
            times.append(frame_idx / fps)
            # 추출한 프레임이 영상의 몇 초 지점에 있는지 저장 
            next_second += 1

        frame_idx += 1

finally:
    cap.release()

print(f"FPS: {fps:.2f}")
print(f"추출한 프레임: {len(frames)}개")

In [ ]:
# 추출한 프레임 확인하기
for sec, frame in zip(times[:5], frames[:5]):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8,4))
    plt.imshow(rgb)
    plt.title(f"{sec:.2f}초")
    plt.axis("off")
    plt.show()

In [ ]:
fig, axes = plt.subplots(3,5,figsize=(20,10))

for ax in axes.flat:
    ax.axis("off")

for ax, frame, sec in zip(axes.flat, frames[:15], times[:15]):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    ax.imshow(rgb)
    ax.set_title(f"{sec:.1f}초")

plt.tight_layout()
plt.show()

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

In [ ]:
import pandas as pd
from collections import Counter

vehicle_names={"car","motorcycle","bus","truck"}

vehicle_ids = [
    class_id for class_id, name in model.names.items() if name in vehicle_names
]

detections = []
counts = []
results = []

for sample_idx, (frame, sec) in enumerate(zip(frames, times)):
    result = model.predict( 
        source=frame,
        classes=vehicle_ids,
        conf=0.25,  # 신뢰도 점수가 0.25 미만인 검출 제외 
        verbose=False,
    )[0]  # 이미지 한장의 추론 결과 

    results.append(result)
    class_counts=Counter()

    for box in result.boxes:  # boxes는 검출된 객체들 
        class_id=int(box.cls.item()) # cls는 예측한 범주 인덱스  
        class_name=result.names[class_id] 
        confidence=float(box.conf.item()) # conf는 검출 신뢰도 점수 

        x1, y1, x2, y2 = box.xyxy[0].cpu().tolist() # xyxy는 박스의 왼쪽 위, 오른쪽 아래 

        class_counts[class_name] += 1

        detections.append({
            "sample_idx": sample_idx,
            "time_sec": sec,
            "vehicle_type": class_name,
            "confidence": confidence,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

    counts.append({
        "sample_idx": sample_idx,
        "time_sec": sec,
        "car": class_counts["car"],
        "motorcycle": class_counts["motorcycle"],
        "bus": class_counts["bus"],
        "truck": class_counts["truck"],
        "total": sum(class_counts.values())
    })


detections_df = pd.DataFrame(
    detections, 
    columns=[
        "sample_idx", "time_sec", "vehicle_type", "confidence", "x1", "y1", "x2", "y2"
    ],
)

counts_df = pd.DataFrame(counts)

display(detections_df.head(10))
display(counts_df)

In [ ]:
fig, axes = plt.subplots(11,1,figsize=(100,50))

for ax in axes.flat:
    ax.axis("off")

for ax, result, time_sec in zip(axes.flat, results[:15], times[:15]):
    annotated = result.plot()

    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{time_sec:.1f}초 | 검출 {len(result.boxes)}대")

plt.tight_layout()
plt.show()

In [ ]:
summary_df = counts_df.copy()

summary_df["video_name"] = Path(video_path).name

summary_df["frame_idx"] = (
    summary_df["time_sec"] * fps
).round().astype(int)

summary_df = summary_df.rename(
    columns={"total": "vehicle_count"}
)
summary_df = summary_df[
    [
        "video_name",
        "frame_idx",
        "time_sec",
        "vehicle_count",
        "car",
        "motorcycle",
        "bus",
        "truck",
    ]
]
display(summary_df)

In [ ]:
output_path = Path("/kaggle/working/vehicle_summary.csv")

summary_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"저장 완료: {output_path}")

# 분석 결과를 나중에도 다시 쓰기 위해 저장

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO
from collections import defaultdict, deque

# 새 객체로 시작해서 이전 추적 상태가 섞이지 않도록 하기
tracking_model = YOLO("yolo11n.pt")

vehicle_names = {"car", "motorcycle", "bus", "truck"}

vehicle_ids = [
    class_id
    for class_id, name in tracking_model.names.items()
    if name in vehicle_names
]

In [ ]:
# 해당 영상의 FPS 확인
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("영상을 열 수 없어.")

tracking_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

if not np.isfinite(tracking_fps) or tracking_fps <= 0:
    raise ValueError("영상 FPS를 확인할 수 없어.")

# 차량 ID별 최근 30개 중심 좌표
track_history = defaultdict(lambda: deque(maxlen=30))

tracking_rows = []
tracking_previews = []
next_second = 0

# 영상 파일을 순서대로 읽으며 추적
tracking_results = tracking_model.track(
    source=str(video_path),
    tracker="botsort.yaml",
    classes=vehicle_ids,
    conf=0.25,
    vid_stride=1,       # 프레임을 건너뛰지 않기
    stream=True,       # 결과를 한 프레임씩 반환
    verbose=False,
)

for frame_idx, result in enumerate(tracking_results):
    time_sec = frame_idx / tracking_fps

    # 차량 박스와 ID가 그려진 이미지
    canvas = result.plot()

    # 추적 ID가 부여된 객체가 있을 때
    if result.boxes.id is not None:
        boxes = result.boxes.xywh.cpu().numpy()
        track_ids = result.boxes.id.int().cpu().tolist()
        class_ids = result.boxes.cls.int().cpu().tolist()

        for box, track_id, class_id in zip(
            boxes, track_ids, class_ids
        ):
            # xywh는 중심 x, 중심 y, 너비, 높이
            center_x, center_y, width, height = box

            # 같은 ID의 좌표를 계속 이어서 저장
            track_history[track_id].append(
                (float(center_x), float(center_y))
            )

            tracking_rows.append({
                "frame_idx": frame_idx,
                "time_sec": time_sec,
                "track_id": track_id,
                "vehicle_type": result.names[class_id],
                "center_x": float(center_x),
                "center_y": float(center_y),
            })

            # 최근 중심 좌표들을 선으로 연결
            points = np.array(
                track_history[track_id], dtype=np.int32
            ).reshape(-1, 1, 2)

            if len(points) >= 2:
                cv2.polylines(
                    canvas,
                    [points],
                    isClosed=False,
                    color=(0, 255, 255),  # BGR: 노란색
                    thickness=2,
                )

    # 분석은 매 프레임, 미리보기만 약 1초마다 저장
    if (
        len(tracking_previews) < 15
        and frame_idx >= round(next_second * tracking_fps)
    ):
        # 미리보기 메모리 사용량 줄이기
        h, w = canvas.shape[:2]
        scale = min(1.0, 640 / w)

        preview = cv2.resize(
            canvas,
            (round(w * scale), round(h * scale)),
        )

        tracking_previews.append((time_sec, preview))
        next_second += 1

tracking_df = pd.DataFrame(
    tracking_rows,
    columns=[
        "frame_idx", "time_sec", "track_id",
        "vehicle_type", "center_x", "center_y",
    ],
)

print("추적 완료!")
display(tracking_df.head(10))

In [ ]:
fig, axes = plt.subplots(3,5, figsize=(20,10))

for ax in axes.flat:
    ax.axis("off")

for ax, (time_sec, preview) in zip(
    axes.flat, tracking_previews
):
    ax.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{time_sec:.1f}초")

plt.tight_layout()
plt.show()

In [ ]:
# 추적 결과에 실제로 존재하는 첫 번째 ID 선택
if not tracking_df.empty:
    selected_id = tracking_df["track_id"].iloc[0]

    vehicle_track = tracking_df[
        tracking_df["track_id"] == selected_id
    ]

    print(f"확인할 차량 ID: {selected_id}")
    display(vehicle_track.head(10))
else:
    print("추적 ID가 부여된 차량이 없어.")